# LoLip on a toy robot

This notebook runs the paper's whole method on a tiny problem, from scratch, in a few minutes. No robotics or machine
learning background is assumed. Run the cells from top to bottom (in Colab: **Runtime → Change runtime type → GPU**,
then **Runtime → Run all**).

**The question.** You have a robot policy that learned many skills from demonstrations, but it does not know which skill
you want. You want to teach it, from only a handful of new demonstrations, to do the one you ask for, **without
retraining it**. The paper does this with a method called **DSBC**. This notebook shows that DSBC barely works on a policy
trained the usual way, and works very well on the same policy trained with our one-line regularizer, **LoLip**.

In [ ]:
import math, torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("running on:", dev)

## 1. The toy world

* **The robot** is a dot that starts at the centre of a table.
* **The targets.** There are 8 targets on a circle around it, numbered 0 to 7.
* **The arrangement.** The circle of targets is not always in the same place: it can be turned to one of 4 positions
  (arrangements 0 to 3). Each arrangement is the same 8 targets, turned by a small angle.
  **The robot sees which arrangement it is in.** This is its *observation*: it tells the robot where the 8 targets are
  right now.
* **The task** is "go to target *k*". There are 8 tasks, one per target.
* **The action.** The robot answers with a short path: 8 positions, from the centre out to a target. Each position is
  (x, y), so one action is a list of 16 numbers.
* **Success** means the path ends at the target that was asked for.

Run the next cell to see the 4 arrangements and a few demonstrations (paths recorded by a person).

In [ ]:
K = 8                 # number of targets (= number of tasks)
H = 8                 # positions in one path
D = 2 * H             # numbers in one action: 8 positions x (x, y) = 16
ARRANGEMENTS = 4
RADIUS = 1.5          # distance of the targets from the centre
SIGMA = 0.08          # how much a person's path wobbles
COLORS = plt.cm.tab10(np.arange(K))

def targets(arrangement):
    # (x, y) of the 8 targets, for each arrangement in the batch: shape (n, 8, 2)
    angle = arrangement[:, None] * 2 * math.pi / (ARRANGEMENTS * K) + 2 * math.pi * torch.arange(K)[None] / K
    return torch.stack([RADIUS * torch.cos(angle), RADIUS * torch.sin(angle)], -1)

def observe(arrangement):
    # what the robot sees: which arrangement it is in (as a one-hot vector of length 4)
    return nn.functional.one_hot(arrangement, ARRANGEMENTS).float()

def demonstrations(n, g=None):
    # n recorded paths: a random arrangement, a random task, and a straight path from the centre to that target (+ wobble)
    arrangement = torch.randint(0, ARRANGEMENTS, (n,), generator=g)
    task = torch.randint(0, K, (n,), generator=g)
    goal = targets(arrangement)[torch.arange(n), task]                       # (n, 2)
    steps = torch.arange(1, H + 1).float()[None, :, None] / H                # 1/8, 2/8, ..., 8/8 of the way
    path = (steps * goal[:, None]).reshape(n, D) + SIGMA * torch.randn(n, D, generator=g)
    return arrangement, task, path

def end_point(path):
    return path[:, -2:]

def success(path, arrangement, task):
    # 1 if the path ends within 3 wobbles of the asked target, else 0
    goal = targets(arrangement)[torch.arange(len(task)), task]
    return (torch.norm(end_point(path).cpu() - goal, dim=1) < 3 * SIGMA).float()

def draw_world(ax, arrangement, title=""):
    T = targets(torch.tensor([arrangement]))[0]
    ax.scatter(T[:, 0], T[:, 1], s=260, c=COLORS, zorder=3)
    for k in range(K): ax.annotate(str(k), T[k], ha="center", va="center", fontsize=9, color="w", zorder=4)
    ax.scatter([0], [0], s=60, c="k", marker="s", zorder=3)
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10)

def draw_paths(ax, path, color):
    p = path.reshape(len(path), H, 2).cpu()
    for q in p: ax.plot(np.r_[0, q[:, 0]], np.r_[0, q[:, 1]], "-", c=color, lw=1, alpha=.7)

g = torch.Generator().manual_seed(0)
fig, ax = plt.subplots(1, ARRANGEMENTS, figsize=(3.3 * ARRANGEMENTS, 3.5))
for a in range(ARRANGEMENTS):
    draw_world(ax[a], a, f"arrangement {a}")
    for k in (0, 3, 6):                                                     # one demonstration each for targets 0, 3, 6
        arr, task, path = demonstrations(200, g)
        i = ((arr == a) & (task == k)).nonzero()[0]
        draw_paths(ax[a], path[i], COLORS[k])
fig.suptitle("The 8 numbered targets in each of the 4 arrangements (black square: the robot's start). "
             "Lines: one demonstration each for targets 0, 3 and 6.", fontsize=10)
plt.tight_layout(); plt.show()

## 2. The plan, in plain words

**Step 1, pretraining.** We collect 256 demonstrations of *all* the tasks mixed together, and train a policy on them.
The policy sees the arrangement, but **we never tell it which target each demonstration went to**. So it learns "from
the centre, go to one of the 8 targets", and picks one at random each time. This is the *base policy*. Asked for target
3, it reaches it only about 1 time in 8 (chance).

We train this base policy twice, on exactly the same data:
* **Vanilla CFM**: the standard way (flow matching, explained below).
* **LoLip**: the same, plus our regularizer.

**Step 2, steering with DSBC.** Now we want to *ask* the base policy for a specific target. We are given only 10 new
demonstrations per target (80 in total) and we do not change the base policy at all. Instead, we train a second, small
model that learns *what to feed into* the base policy so that it goes where we ask. Section 5 explains how.

**The result to watch:** DSBC on the Vanilla base fails; DSBC on the LoLip base succeeds almost every time.

## 3. What a flow policy is

A flow policy makes an action out of random noise.

1. Draw a random **noise** `z`: 16 random numbers from a bell curve (a standard Gaussian), the same size as an action.
2. Move it, little by little, in 50 small steps. Each step is decided by a neural network `v` (the *velocity*), which
   looks at where the point is now, how far along it is (`t` from 0 to 1), and the observation.
3. After the last step, the point is the **action** `a` (a path).

Different noises give different actions, so the noise is how the policy chooses *which* of the 8 targets to go to.
Because every step is small, the same steps can be **run backwards**: given an action, we can recover the noise that
produces it. DSBC relies on this.

**Training (flow matching).** Take a demonstration `a` and a random noise `z`. Pick a point part of the way between them,
`x_t = (1 - t) z + t a`. Train `v` to point from `z` towards `a`, that is, to predict `a - z`. Do this many times.

**LoLip** adds one term. A small step of the policy can squeeze nearby points together (it *contracts*). Vanilla flow
matching squeezes a lot: a large cloud of noises ends up packed onto each tightly clustered target. LoLip measures how much
each step shrinks a random direction `u` and penalizes it if it shrinks by more than a set amount, so the whole policy never
squeezes any direction by more than a factor `T`. In the code it is the function `lolip_penalty`, four lines long.

In [ ]:
class Flow(nn.Module):
    # the velocity network v(x, t | condition): a small 3-layer neural network
    def __init__(self, dim, cond, h=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim + 1 + cond, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, dim))
    def forward(self, x, t, c):
        return self.net(torch.cat([x, t[:, None], c], 1))

N = 50                                            # number of small steps from noise to action

def generate(v, z, c):                            # noise -> action
    x = z
    for i in range(N):
        x = x + v(x, torch.full((len(x),), i / N, device=x.device), c) / N
    return x

def invert(v, a, c):                              # action -> noise (the same steps, run backwards)
    x = a
    for i in reversed(range(N)):
        x = x - v(x, torch.full((len(x),), i / N, device=x.device), c) / N
    return x

def lolip_penalty(v, x_t, t, c, T):
    s_min = T ** (-1 / N)                                         # the least a single step may shrink a direction
    u = torch.randn_like(x_t); u = u / u.norm(dim=1, keepdim=True) # a random direction of length 1
    step = lambda x: x + v(x, t, c) / N                           # one small step
    _, Ju = torch.autograd.functional.jvp(step, x_t, u, create_graph=True)   # where the step sends that direction
    return 0.5 * torch.relu(s_min - Ju.norm(dim=1)).mean()        # penalize it only if it got shorter than s_min

def flow_matching(data, cond, lam=0.0, T=10.0, steps=15000, seed=0):
    # trains a flow policy on (data, cond) pairs; lam = 0 is Vanilla CFM, lam > 0 adds LoLip
    torch.manual_seed(seed)
    data, cond = data.to(dev), cond.to(dev)
    v = Flow(data.shape[1], cond.shape[1]).to(dev)
    opt = torch.optim.Adam(v.parameters(), 2e-3)
    for _ in range(steps):
        i = torch.randint(0, len(data), (512,), device=dev)
        a, c = data[i], cond[i]
        z = torch.randn_like(a); t = torch.rand(len(a), device=dev)
        x_t = (1 - t[:, None]) * z + t[:, None] * a
        loss = ((v(x_t, t, c) - (a - z)) ** 2).sum(1).mean()      # flow matching
        if lam > 0:
            loss = loss + lam * lolip_penalty(v, x_t, t, c, T)    # + LoLip
        opt.zero_grad(); loss.backward(); opt.step()
    return v.eval()

## 4. Step 1: pretrain the two base policies

Same 256 demonstrations, same network, same training. The only difference: LoLip adds the penalty (weight λ = 50,
largest allowed squeeze T = 10). Each takes about half a minute on a GPU.

Then we let each base policy act on its own, 40 times in arrangement 0. Both go to *some* target, picked at random:
neither knows which one we want.

In [ ]:
g = torch.Generator().manual_seed(0)
base_arr, base_task, base_paths = demonstrations(256, g)        # pretraining data (the task labels are never used)

vanilla = flow_matching(base_paths, observe(base_arr))
lolip = flow_matching(base_paths, observe(base_arr), lam=50.0, T=10.0)

fig, ax = plt.subplots(1, 2, figsize=(7, 3.6))
arr0 = torch.zeros(40, dtype=torch.long)
with torch.no_grad():
    for x, (name, base) in zip(ax, [("Vanilla CFM", vanilla), ("LoLip", lolip)]):
        draw_world(x, 0, f"{name} base policy, on its own")
        draw_paths(x, generate(base, torch.randn(40, D, device=dev), observe(arr0).to(dev)), "k")
plt.tight_layout(); plt.show()

## 5. Step 2: steer each base policy with DSBC

We now get **10 new demonstrations per target** (80 in total), each labelled with its arrangement and its target.
The base policy is **frozen**: its weights never change. DSBC works in three steps:

1. **Find the noise behind each new demonstration.** Run the base policy *backwards* on the demonstration. The result is
   the noise that the base policy would turn into that exact path.
2. **Train a noise policy.** A second small flow policy learns to produce those noises. It sees the arrangement **and the
   asked target**: "in arrangement 2, for target 5, give a noise like these".
3. **Use it.** To reach target *k*: ask the noise policy for a noise, then feed that noise into the base policy, which
   turns it into a path.

We test on 2000 random (arrangement, target) requests, and also count how often the base policy alone, fed plain random
noise, happens to reach the asked target.

In [ ]:
dsbc_arr, dsbc_task, dsbc_paths = demonstrations(10 * K, g)      # the 80 new demonstrations, used only by DSBC

def task_and_obs(arrangement, task):                             # what the noise policy sees
    return torch.cat([observe(arrangement), nn.functional.one_hot(task, K).float()], 1)

def dsbc(base, seed=0, n=2000):
    # 1. the noise behind each new demonstration
    with torch.no_grad():
        w = invert(base, dsbc_paths.to(dev), observe(dsbc_arr).to(dev))
    # 2. a noise policy that produces those noises, given the arrangement and the asked target
    noise_policy = flow_matching(w, task_and_obs(dsbc_arr, dsbc_task), steps=8000, seed=seed)
    # 3. test it on n random requests
    gt = torch.Generator().manual_seed(1)
    arr = torch.randint(0, ARRANGEMENTS, (n,), generator=gt); task = torch.randint(0, K, (n,), generator=gt)
    with torch.no_grad():
        alone = generate(base, torch.randn(n, D, device=dev), observe(arr).to(dev))
        z = generate(noise_policy, torch.randn(n, D, device=dev), task_and_obs(arr, task).to(dev))
        steered = generate(base, z, observe(arr).to(dev))
    return dict(w=w.cpu(), arr=arr, task=task, steered=steered.cpu(),
                base_success=success(alone, arr, task).mean().item(),
                dsbc_success=success(steered, arr, task).mean().item())

results = {"Vanilla CFM": dsbc(vanilla), "LoLip": dsbc(lolip)}
print(f"reaching the asked target (chance = 1 in {K} = {1 / K:.0%}):")
for name, r in results.items():
    print(f"  {name:12} base policy alone: {r['base_success']:4.0%}    with DSBC: {r['dsbc_success']:4.0%}")

## 6. The results, in pictures

For each base policy (one row each):

* **Left: success.** How often the path ends at the asked target: the base policy alone, and with DSBC.
* **Middle: the noises DSBC found** in step 1, one dot per new demonstration, coloured by its target. A noise is 16
  numbers; we plot the first two. Plain random noise falls outside the dotted circle only about 1 time in 100, and the
  base policy was only ever trained on noise like that.
* **Right: DSBC in action.** Asked for target 0 in arrangement 0, 40 times: the paths it produces.

In [ ]:
def show(results):
    fig, ax = plt.subplots(len(results), 3, figsize=(12, 3.9 * len(results)), squeeze=False)
    for row, (name, r) in zip(ax, results.items()):
        s = [r["base_success"], r["dsbc_success"]]
        row[0].bar(["base policy alone", "with DSBC"], s, color=["#b4b4b8", "#d91e5b"])
        row[0].axhline(1 / K, ls=":", c="k", lw=1); row[0].text(1.45, 1 / K, "chance", va="bottom", ha="right", fontsize=8)
        for i, v in enumerate(s): row[0].text(i, v + .02, f"{v:.0%}", ha="center", fontsize=10)
        row[0].set_ylim(0, 1.08); row[0].set_title(f"{name}: reaches the asked target", fontsize=10)

        p = r["w"][:, :2]
        outside = int((p.norm(dim=1) > 3).sum())
        row[1].scatter(p[:, 0], p[:, 1], s=14, c=COLORS[dsbc_task])
        row[1].add_patch(plt.Circle((0, 0), 3, fill=False, ls=":"))
        off = int((p.abs() > 12).any(1).sum())
        row[1].set_xlim(-12, 12); row[1].set_ylim(-12, 12); row[1].set_aspect("equal")
        row[1].set_title(f"noises behind the 80 new demonstrations\n{outside} of 80 outside the circle"
                         + (f" ({off} beyond the edges of the plot)" if off else ""), fontsize=10)

        i = ((r["arr"] == 0) & (r["task"] == 0)).nonzero()[:40, 0]
        draw_world(row[2], 0, f"DSBC asked for target 0 (40 tries)")
        draw_paths(row[2], r["steered"][i], COLORS[0])
    plt.tight_layout(); plt.show()

show(results)

## 7. Why

**The base policy alone** does not know which target we want, so it reaches the asked one about 1 time in 8, with either
training.

**Vanilla CFM** squeezes a large region of noise onto each target, because the demonstrations around each target are
tightly clustered. Run backwards, that squeeze becomes a stretch: two demonstrations that differ by a tiny wobble map to
noises far apart, many of them far outside the range of normal random noise (middle plot, top row). The noise policy has
only 10 such scattered points per target to learn from, so the noises it produces are not the ones that lead to the asked
target, and DSBC fails.

**LoLip** limits how much the policy may squeeze. Run backwards, it stretches little: the new demonstrations map to noises
that look like normal random noise and sit together by target (middle plot, bottom row). The noise policy learns them
easily, and DSBC reaches the asked target almost every time.

The price: the LoLip base policy has a wider spread of actions when fed plain random noise. In the paper, this lowers the
base policy's own success somewhat, and DSBC recovers it.

## 8. Try it yourself

Change λ (how strongly the penalty counts) and T (how much squeeze is allowed in total), then press **Train**. λ = 0 is
Vanilla CFM. Each run trains a new base policy and runs DSBC on it (about a minute on a GPU).

Things to try:
* A small λ (0.5 or 5): how much regularization does DSBC need?
* At λ = 50, a strict T = 2 versus a loose T = 100.

In [ ]:
def try_lolip(lam, T):
    base = flow_matching(base_paths, observe(base_arr), lam=lam, T=T)
    show({"Vanilla CFM": results["Vanilla CFM"], f"LoLip (λ = {lam:g}, T = {T:g})": dsbc(base)})

try:
    import ipywidgets as W
    from IPython.display import display, clear_output
    lam_w = W.FloatLogSlider(value=50, base=10, min=-1, max=2.5, description="λ")
    T_w = W.FloatLogSlider(value=10, base=10, min=0.3, max=2, description="T")
    button = W.Button(description="Train", button_style="primary"); out = W.Output()
    def on_click(_):
        with out:
            clear_output(wait=True); print("training ...")
            lam, T = lam_w.value, T_w.value
            clear_output(wait=True); try_lolip(lam, T)
    button.on_click(on_click)
    display(W.HBox([lam_w, T_w, button]), out)
except ImportError:
    print("ipywidgets is not installed; call try_lolip(lam, T) directly, for example:")
    try_lolip(5.0, 10.0)